In [1]:
import os
import pandas as pd
from datetime import datetime
from transformers import pipeline

month_to_spanish = {
    1: "enero",
    2: "febrero",
    3: "marzo",
    4: "abril",
    5: "mayo",
    6: "junio",
    7: "julio",
    8: "agosto",
    9: "septiembre",
    10: "octubre",
    11: "noviembre",
    12: "diciembre"
}

def import_mananeras(base_path, year, month_number, day, all_man = False,
                     file_name = 'csv_por_participante/PRESIDENTE ANDRES MANUEL LOPEZ OBRADOR.csv' ):
    
    """
    Imports files from mananeras folder

    Args:
        base_path (str): The root path containing year/month-year/month day, year folders.
        year (int): year where you want to extract info
        month (int): month from where you want to extract info
        file_name (str): Name of the specific file to import. 
        all (bool): if True, imports the whole mananera

    Returns:
        list of pandas.DataFrame: List of DataFrames for the files imported.
    """
    
    month_to_spanish = {
    1: "enero",
    2: "febrero",
    3: "marzo",
    4: "abril",
    5: "mayo",
    6: "junio",
    7: "julio",
    8: "agosto",
    9: "septiembre",
    10: "octubre",
    11: "noviembre",
    12: "diciembre"}

    month_sp = month_to_spanish[month_number]
    if all_man == True: 
        file_name = f'{base_path}{year}/{month_number}-{year}/{month_sp} {day}, {year}/mananera_{day:02}_{month_number:02}_{year}.csv'
    else: 
        filename = file_name = f'{base_path}{year}/{month_number}-{year}/{month_sp} {day}, {year}/{file_name}'

    try:
        df = pd.read_csv(file_name)
    except: 
        print('No conference that day')
        df = pd.DataFrame()

    return df





/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
base_path = '../../data/02-conferences/raw/'

final = pd.DataFrame()
for y in range(2021, 2024): 
    int1 = pd.DataFrame()
    for mmm in range(1, 13): 
        int2 = pd.DataFrame()
        for ddd in range(1, 31): 
            df = import_mananeras(base_path, y, mmm, ddd, 
                      file_name='csv_por_participante/PREGUNTA.CSV')
            try:
                df = pd.DataFrame({'Texto': [" ".join(df['Texto'])]})
                df['month'] = mmm
                df['ddd'] = ddd
                df['y'] = y
            except:
                df = pd.DataFrame()
            

            int2 = pd.concat([int2, df])
        int1 = pd.concat([int1, int2])
    final = pd.concat([final, int1]).reset_index(drop=True)





No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conferen

In [50]:
import re

exclude_phrases = [
    "Hugo López Gatell", "Ciudad de México", "México", 
    "Presidente", "COVID", "Gobierno", 'Si', 'Y', 'También',
    'Gracias', 'Buenos', 'Cómo', 'Ayer', 'Se', 'Qué', 'Yo',
    'Preguntarle', 'Ahora', 'Una', 'QR', 'Cuál', 'Comisión', 
    'Federal'
]

# Function to filter sentences containing "días" and their adjacent ones
def filter_sentences(text):
    sentences = re.split(r'(?<!\w\.\w.)(?<![A-Z][a-z]\.)(?<=\.|\?)\s', text)
    result = []
    for i, sentence in enumerate(sentences):
        if 'días' in sentence:
            result.append(sentence)
            if i > 0:  # Add the previous sentence if it exists
                result.append(sentences[i-1])
            if i < len(sentences) - 1:  # Add the next sentence if it exists
                result.append(sentences[i+1])
    # Return only the relevant sentences joined together
    return ' '.join(set(result))

# Apply the function to extract relevant sentences
final['filtered_sentences'] = final['Texto'].apply(filter_sentences)

# Function to clean and filter the extracted sentences
def filter_text(text):
    # Remove excluded phrases
    for phrase in exclude_phrases:
        text = text.replace(phrase, "")
    # Remove extra spaces created during replacement
    text = re.sub(r'\s+', ' ', text).strip()
    # Filter words containing uppercase letters
    filtered_words = re.findall(r'\b\w*[A-Z]\w*\b', text)
    return ' '.join(filtered_words)

# Apply the function to the 'text' column
final['text_aux'] = final['Texto'].apply(filter_text)

In [53]:
final['Texto'][2]

' Presidente, buenos días. Judith Sánchez Reyes, corresponsal de Imagen del Golfo, de Veracruz. Presidente, la Comisión Federal de Electricidad acaba de reconocer que el documento que se presentó precisamente para dar a conocer las causas que provocaron el apagón masivo el año pasado sí es falso, tal como se había señalado en su momento por el gobierno de Tamaulipas; de hecho, el propio titular está ya haciendo las investigaciones pertinentes. ¿Cuál es su opinión al respecto? Y si en esta situación, al ser un documento falso que, vaya, supuestamente se daban ahí las causas de este apagón, será conveniente nuevamente hacer investigaciones y poder saber realmente qué es lo que sucedió con este apagón y a lo mejor pensar si pudiera repetirse nuevamente.  En otro tema, presidente, el año pasado obviamente uno de los casos que más llamó la atención mediáticamente y, bueno, a nivel opinión pública fue justamente la extradición de Emilio Lozoya. Llegó y llegó aquí a nuestro país; sin embargo,

In [54]:
import pandas as pd
import numpy as np
import openai
import time

from openai import OpenAI
api_key = 'REDACTED_OPENAI_API_KEY'
openai.api_key = api_key

In [56]:
final['Texto'][1]

' Buenos días, presidente; buenos días, subsecretario. Shaila Rosagel, corresponsal de Grupo Healy, El Imparcial, de Sonora; La Crónica, de Mexicali; y Frontera, de Tijuana. Preguntarle, presidente: ¿cuándo van a llegar las vacunas contra COVID-19 a Baja California y a Sonora? Si ya tienen delimitado esto. ¿Y cuántas será y a quienes se les van a aplicar estas primeras dosis? También si ya se les dio a conocer a los gobernadores cuándo llegaran las vacunas a cada una de sus entidades o si apenas se va a hablar de ello.  Gracias, presidente. Una segunda pregunta…  Presidente, en una segunda pregunta, el gobierno de Sonora dio a conocer que planea adquirir la vacuna y que está en pláticas con Pfizer, con Moderna y con AstraZeneca. ¿Cuál es su postura sobre este plan del gobierno de Sonora? Y si autorizará la Cofepris estas adquisiciones que proyecta hacer Sonora y cuándo se daría esa autorización.  Presidente, preguntarle también, no solamente se ha conocido del caso del médico que vacun

In [55]:
pred = []
prompt_list = [final['Texto'][1], final['Texto'][2]]
for i in prompt_list:
    completion = openai.chat.completions.create(model="gpt-3.5-turbo",
                    messages=[
                        {"role": "system", "content": "Eres un asistente de investigación para un proyecto que busca analizar las conferencias de prensa de AMLO. Te dare el texto de todas las preguntas y conversación de los reporteros que fueron a la conferencia, regularmente estos se presentan, dicen su nombre y el medio al que pertenecen. Necesito que extraigas el nombre de todos los periodistas que lo mencionen, y el medio. Ponlos asi: Nombre, Medio | Nombre2, Medio2"},
                        {"role": "user", "content": i}
                        ]
                )
    rev = completion.choices[0].message.content.strip()
    print(rev)

    pred.append(rev)
    time.sleep(1)

Shaila Rosagel, Grupo Healy, El Imparcial, La Crónica, Frontera | Sheila, no se menciona el medio | no mencionado, no mencionado | no mencionado, no mencionado
Judith Sánchez Reyes, Imagen del Golfo | Pedro Villa y Caña, El Universal | Diego Elías Cedillo, diario Basta, Grupo Cantón | Meme Yamel, The México News y Sin Censura


In [64]:
pred1 = []
for i in range(0, 10):
    texto = final['Texto'][i]
    completion = openai.chat.completions.create(model="gpt-3.5-turbo",
                    messages=[
                        {"role": "system", "content": "Eres un asistente de investigación para un proyecto que busca analizar las conferencias de prensa de AMLO. Te dare el texto de todas las preguntas y conversación de los reporteros que fueron a la conferencia, regularmente estos se presentan, dicen su nombre y el medio al que pertenecen. Necesito que extraigas el nombre de todos los periodistas que lo mencionen, y el medio. Ponlos asi: Nombre, Medio | Nombre2, Medio2"},
                        {"role": "user", "content": texto}
                        ]
                )
    rev = completion.choices[0].message.content.strip()
    print(rev)

    pred1.append(rev)
    time.sleep(1)

Héctor Tlatempa, Puntos Suspensivos Radio, Puntos Suspensivos Comunicación | Demian Duarte, Sonora Power, Lobos FM y Política y RockandRoll Radio | Nuri Fernández, La Caracola | Meme Yamel, The México News y Sin Censura | Carlos Calzada, Radio Educación | Hans Salazar, Noticiero en Redes.
Shaila Rosagel, Grupo Healy, El Imparcial, de Sonora; La Crónica, de Mexicali; y Frontera, de Tijuana | Sheila,  | Sheila,  |
Judith Sánchez Reyes, Imagen del Golfo | Pedro Villa y Caña, El Universal | Diego Elías Cedillo, diario Basta, Grupo Cantón | Meme Yamel, The México News y Sin Censura
Shaila Rosagel, Grupo Healy; El Imparcial; La Crónica; Frontera | Daniel Marmolejo, La Cuarta República; Voces del Periodista | Judith Sánchez Reyes, Imagen del Golfo | Marco Antonio Olvera, Énfasis; Marco Olvera Oficial
Paul Velázquez, Los Mochis | Francisco Huerta, (Inaudible)
Judith Sánchez Reyes, Imagen del Golfo | Meme Yamel, The Mexico News y Sin Censura
Lizbeth Álvarez, Gurú Político y ZMG Noticias | Carlo

In [65]:
pred = []
for i in range(10, len(final)+1):
    texto = final['Texto'][i]
    completion = openai.chat.completions.create(model="gpt-3.5-turbo",
                    messages=[
                        {"role": "system", "content": "Eres un asistente de investigación para un proyecto que busca analizar las conferencias de prensa de AMLO. Te dare el texto de todas las preguntas y conversación de los reporteros que fueron a la conferencia, regularmente estos se presentan, dicen su nombre y el medio al que pertenecen. Necesito que extraigas el nombre de todos los periodistas que lo mencionen, y el medio. Ponlos asi: Nombre, Medio | Nombre2, Medio2"},
                        {"role": "user", "content": texto}
                        ]
                )
    rev = completion.choices[0].message.content.strip()
    print(i)

    pred.append(rev)

10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
277
278
279
280
281
28

BadRequestError: Error code: 400 - {'error': {'message': "This model's maximum context length is 16385 tokens. However, your messages resulted in 17909 tokens. Please reduce the length of the messages.", 'type': 'invalid_request_error', 'param': 'messages', 'code': 'context_length_exceeded'}}

In [68]:
pred7 = ['Not available']

In [69]:
pred3 = []
for i in range(537, len(final)):
    texto = final['Texto'][i]
    try: 
        completion = openai.chat.completions.create(model="gpt-3.5-turbo",
                    messages=[
                        {"role": "system", "content": "Eres un asistente de investigación para un proyecto que busca analizar las conferencias de prensa de AMLO. Te dare el texto de todas las preguntas y conversación de los reporteros que fueron a la conferencia, regularmente estos se presentan, dicen su nombre y el medio al que pertenecen. Necesito que extraigas el nombre de todos los periodistas que lo mencionen, y el medio. Ponlos asi: Nombre, Medio | Nombre2, Medio2"},
                        {"role": "user", "content": texto}
                        ]
                )
        rev = completion.choices[0].message.content.strip()
        print(i)
    except: 
        rev = 'Not available'
        print('Not available')

    pred3.append(rev)

Not available
538
539
540
541
542
543
544
545
546
547
548
549
550
551
552
553
554
555
556
557
558
559
560
561
562
563
564
565
566
567
568
569
570
571
572
573
574
575
576
577
578
579
580
581
582
583
584
585
586
587
588
589
590
591
592
593
594
595
596
597
598
599
600
601
602
603
604
605
606
607
608
609
610
611
612
613
614
615
616
617
618
619
620
Not available
622
623
624
625
626
627
628
629
630
631
632
633
634
635
636
637
638
639
640
641
642
643
644
645
646
647
648
649
650
651
652
653
654


KeyError: 655

In [71]:
pred1

['Héctor Tlatempa, Puntos Suspensivos Radio, Puntos Suspensivos Comunicación | Demian Duarte, Sonora Power, Lobos FM y Política y RockandRoll Radio | Nuri Fernández, La Caracola | Meme Yamel, The México News y Sin Censura | Carlos Calzada, Radio Educación | Hans Salazar, Noticiero en Redes.',
 'Shaila Rosagel, Grupo Healy, El Imparcial, de Sonora; La Crónica, de Mexicali; y Frontera, de Tijuana | Sheila,  | Sheila,  |',
 'Judith Sánchez Reyes, Imagen del Golfo | Pedro Villa y Caña, El Universal | Diego Elías Cedillo, diario Basta, Grupo Cantón | Meme Yamel, The México News y Sin Censura',
 'Shaila Rosagel, Grupo Healy; El Imparcial; La Crónica; Frontera | Daniel Marmolejo, La Cuarta República; Voces del Periodista | Judith Sánchez Reyes, Imagen del Golfo | Marco Antonio Olvera, Énfasis; Marco Olvera Oficial',
 'Paul Velázquez, Los Mochis | Francisco Huerta, (Inaudible)',
 'Judith Sánchez Reyes, Imagen del Golfo | Meme Yamel, The Mexico News y Sin Censura',
 'Lizbeth Álvarez, Gurú Polít

In [72]:
pred_final = pred1 + pred + pred7 + pred3

len(pred_final)

655

In [73]:
len(final)

655

In [74]:
final['periodistas'] = pred_final
final

,Texto,month,ddd,y,text_aux,filtered_sentences,periodistas
0,La Secretaría de la Defensa Nacional informa ...,1,4,2021,La Defensa Nacional Aeropuerto Internacional F...,"Buenos días a todas y a todos. Sí, buenos día...","Héctor Tlatempa, Puntos Suspensivos Radio, Pun..."
1,"Buenos días, presidente; buenos días, subsecr...",1,5,2021,Shaila Rosagel Grupo Healy El Imparcial Sonora...,"Shaila Rosagel, corresponsal de Grupo Healy, E...","Shaila Rosagel, Grupo Healy, El Imparcial, de ..."
2,"Presidente, buenos días. Judith Sánchez Reyes...",1,6,2021,Judith Sánchez Reyes Imagen Golfo Veracruz Ele...,"Judith Sánchez Reyes, corresponsal de Imagen d...","Judith Sánchez Reyes, Imagen del Golfo | Pedro..."
3,"Buenos días, presidente. Shaila Rosagel, corr...",1,8,2021,Shaila Rosagel Grupo Healy El Imparcial Sonora...,"Gracias. Muchas gracias. Presidente, buenos ...","Shaila Rosagel, Grupo Healy; El Imparcial; La ..."
4,La Secretaría de la Defensa Nacional informa ...,1,11,2021,La Defensa Nacional Aeropuerto Internacional F...,Si usted en particular va a aplicar a algún t...,"Paul Velázquez, Los Mochis | Francisco Huerta,..."
...,...,...,...,...,...,...,...
650,Gracias. Muy buenos días a todas y a todos. L...,12,19,2023,Muy Liliana Noble Pulso Saludable Time Out Par...,"Muy buenos días a todas y a todos. Gracias, d...","Liliana Noble, Pulso Saludable | Liliana Piña,..."
651,Continúan los esfuerzos del Gobierno de Méxic...,12,20,2023,Continúan Otis Guerrero Estamos A Tianguis Bie...,"Buenos días, señor presidente. ¿Se va a busca...","Andrés García, Códice 21 | Arturo Páramo, Grup..."
652,"La Secretaría de Infraestructura, Comunicacio...",12,21,2023,La Infraestructura Comunicaciones Transportes ...,"Buenos días, señor presidente. Con Agua Salud...","Fernando Olivas, Radio Relax y ¿Qué Pasó?, dig..."
653,"Campaña de miedo en Tabasco. Políticos, comun...",12,27,2023,Campaña Tabasco Políticos Villahermosa Tabasco...,"Ernesto Ledesma, de Rompeviento Tv. Jefe de Go...","- José Sobrevilla, Noreste\n- Ernesto Ledesma,..."


In [75]:
final.to_parquet('../../data/02-conferences/auxiliar/periodistas_2021_2023.parquet')

In [80]:
final = pd.read_parquet('../../data/02-conferences/auxiliar/periodistas_2021_2023.parquet')

In [82]:
final['periodistas'] = final['periodistas'].str.replace(' y ', ',', regex=False)
final['periodistas'] = final['periodistas'].str.replace(';', ',', regex=False)
final

,Texto,month,ddd,y,text_aux,filtered_sentences,periodistas
0,La Secretaría de la Defensa Nacional informa ...,1,4,2021,La Defensa Nacional Aeropuerto Internacional F...,"Buenos días a todas y a todos. Sí, buenos día...","Héctor Tlatempa, Puntos Suspensivos Radio, Pun..."
1,"Buenos días, presidente; buenos días, subsecr...",1,5,2021,Shaila Rosagel Grupo Healy El Imparcial Sonora...,"Shaila Rosagel, corresponsal de Grupo Healy, E...","Shaila Rosagel, Grupo Healy, El Imparcial, de ..."
2,"Presidente, buenos días. Judith Sánchez Reyes...",1,6,2021,Judith Sánchez Reyes Imagen Golfo Veracruz Ele...,"Judith Sánchez Reyes, corresponsal de Imagen d...","Judith Sánchez Reyes, Imagen del Golfo | Pedro..."
3,"Buenos días, presidente. Shaila Rosagel, corr...",1,8,2021,Shaila Rosagel Grupo Healy El Imparcial Sonora...,"Gracias. Muchas gracias. Presidente, buenos ...","Shaila Rosagel, Grupo Healy, El Imparcial, La ..."
4,La Secretaría de la Defensa Nacional informa ...,1,11,2021,La Defensa Nacional Aeropuerto Internacional F...,Si usted en particular va a aplicar a algún t...,"Paul Velázquez, Los Mochis | Francisco Huerta,..."
...,...,...,...,...,...,...,...
650,Gracias. Muy buenos días a todas y a todos. L...,12,19,2023,Muy Liliana Noble Pulso Saludable Time Out Par...,"Muy buenos días a todas y a todos. Gracias, d...","Liliana Noble, Pulso Saludable | Liliana Piña,..."
651,Continúan los esfuerzos del Gobierno de Méxic...,12,20,2023,Continúan Otis Guerrero Estamos A Tianguis Bie...,"Buenos días, señor presidente. ¿Se va a busca...","Andrés García, Códice 21 | Arturo Páramo, Grup..."
652,"La Secretaría de Infraestructura, Comunicacio...",12,21,2023,La Infraestructura Comunicaciones Transportes ...,"Buenos días, señor presidente. Con Agua Salud...","Fernando Olivas, Radio Relax,¿Qué Pasó?, digit..."
653,"Campaña de miedo en Tabasco. Políticos, comun...",12,27,2023,Campaña Tabasco Políticos Villahermosa Tabasco...,"Ernesto Ledesma, de Rompeviento Tv. Jefe de Go...","- José Sobrevilla, Noreste\n- Ernesto Ledesma,..."


In [83]:
# Step 1: Split the 'reporters' column into a list of reporters

final['reporters_list'] = final['periodistas'].str.split('|')

# Step 2: Expand the list into separate columns
max_reporters = final['reporters_list'].apply(len).max()  # Find the maximum number of reporters in a single row
reporter_columns = [f"reporter{i+1}" for i in range(max_reporters)]  # Generate column names

final[reporter_columns] = pd.DataFrame(final['reporters_list'].tolist(), index=final.index)

# Step 3: Drop the intermediate 'reporters_list' column if not needed
final = final.drop(columns=['reporters_list'])
final

,Texto,month,ddd,y,text_aux,filtered_sentences,periodistas,reporter1,reporter2,reporter3,...,reporter11,reporter12,reporter13,reporter14,reporter15,reporter16,reporter17,reporter18,reporter19,reporter20
0,La Secretaría de la Defensa Nacional informa ...,1,4,2021,La Defensa Nacional Aeropuerto Internacional F...,"Buenos días a todas y a todos. Sí, buenos día...","Héctor Tlatempa, Puntos Suspensivos Radio, Pun...","Héctor Tlatempa, Puntos Suspensivos Radio, Pun...","Demian Duarte, Sonora Power, Lobos FM,Polític...","Nuri Fernández, La Caracola",...,None,None,None,None,None,None,None,None,None,None
1,"Buenos días, presidente; buenos días, subsecr...",1,5,2021,Shaila Rosagel Grupo Healy El Imparcial Sonora...,"Shaila Rosagel, corresponsal de Grupo Healy, E...","Shaila Rosagel, Grupo Healy, El Imparcial, de ...","Shaila Rosagel, Grupo Healy, El Imparcial, de ...","Sheila,","Sheila,",...,None,None,None,None,None,None,None,None,None,None
2,"Presidente, buenos días. Judith Sánchez Reyes...",1,6,2021,Judith Sánchez Reyes Imagen Golfo Veracruz Ele...,"Judith Sánchez Reyes, corresponsal de Imagen d...","Judith Sánchez Reyes, Imagen del Golfo | Pedro...","Judith Sánchez Reyes, Imagen del Golfo","Pedro Villa,Caña, El Universal","Diego Elías Cedillo, diario Basta, Grupo Cantón",...,None,None,None,None,None,None,None,None,None,None
3,"Buenos días, presidente. Shaila Rosagel, corr...",1,8,2021,Shaila Rosagel Grupo Healy El Imparcial Sonora...,"Gracias. Muchas gracias. Presidente, buenos ...","Shaila Rosagel, Grupo Healy, El Imparcial, La ...","Shaila Rosagel, Grupo Healy, El Imparcial, La ...","Daniel Marmolejo, La Cuarta República, Voces ...","Judith Sánchez Reyes, Imagen del Golfo",...,None,None,None,None,None,None,None,None,None,None
4,La Secretaría de la Defensa Nacional informa ...,1,11,2021,La Defensa Nacional Aeropuerto Internacional F...,Si usted en particular va a aplicar a algún t...,"Paul Velázquez, Los Mochis | Francisco Huerta,...","Paul Velázquez, Los Mochis","Francisco Huerta, (Inaudible)",None,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
650,Gracias. Muy buenos días a todas y a todos. L...,12,19,2023,Muy Liliana Noble Pulso Saludable Time Out Par...,"Muy buenos días a todas y a todos. Gracias, d...","Liliana Noble, Pulso Saludable | Liliana Piña,...","Liliana Noble, Pulso Saludable","Liliana Piña, ATiempo.com.mx,PuenteLibre.mx","Fanny Martínez, La Verdad Noticias",...,None,None,None,None,None,None,None,None,None,None
651,Continúan los esfuerzos del Gobierno de Méxic...,12,20,2023,Continúan Otis Guerrero Estamos A Tianguis Bie...,"Buenos días, señor presidente. ¿Se va a busca...","Andrés García, Códice 21 | Arturo Páramo, Grup...","Andrés García, Códice 21","Arturo Páramo, Grupo Imagen","Ángela Rodríguez, Radio Fórmula",...,None,None,None,None,None,None,None,None,None,None
652,"La Secretaría de Infraestructura, Comunicacio...",12,21,2023,La Infraestructura Comunicaciones Transportes ...,"Buenos días, señor presidente. Con Agua Salud...","Fernando Olivas, Radio Relax,¿Qué Pasó?, digit...","Fernando Olivas, Radio Relax,¿Qué Pasó?, digital",Marisol Cruz Hernández,"Carlos Domínguez, Nación 14",...,None,None,None,None,None,None,None,None,None,None
653,"Campaña de miedo en Tabasco. Políticos, comun...",12,27,2023,Campaña Tabasco Políticos Villahermosa Tabasco...,"Ernesto Ledesma, de Rompeviento Tv. Jefe de Go...","- José Sobrevilla, Noreste\n- Ernesto Ledesma,...","- José Sobrevilla, Noreste\n- Ernesto Ledesma,...",None,None,...,None,None,None,None,None,None,None,None,None,None


In [87]:
final['date'] = (final['y'].astype(str) + '-' + 
                 final['month'].astype(str).str.zfill(2) + 
                 '-' + final['ddd'].astype(str).str.zfill(2))


# Step 2: Convert the string into a proper date variable
final['date'] = pd.to_datetime(final['date'], 
                               format='%Y-%m-%d')
final

,Texto,month,ddd,y,text_aux,filtered_sentences,periodistas,reporter1,reporter2,reporter3,...,reporter12,reporter13,reporter14,reporter15,reporter16,reporter17,reporter18,reporter19,reporter20,date
0,La Secretaría de la Defensa Nacional informa ...,1,4,2021,La Defensa Nacional Aeropuerto Internacional F...,"Buenos días a todas y a todos. Sí, buenos día...","Héctor Tlatempa, Puntos Suspensivos Radio, Pun...","Héctor Tlatempa, Puntos Suspensivos Radio, Pun...","Demian Duarte, Sonora Power, Lobos FM,Polític...","Nuri Fernández, La Caracola",...,None,None,None,None,None,None,None,None,None,2021-01-04
1,"Buenos días, presidente; buenos días, subsecr...",1,5,2021,Shaila Rosagel Grupo Healy El Imparcial Sonora...,"Shaila Rosagel, corresponsal de Grupo Healy, E...","Shaila Rosagel, Grupo Healy, El Imparcial, de ...","Shaila Rosagel, Grupo Healy, El Imparcial, de ...","Sheila,","Sheila,",...,None,None,None,None,None,None,None,None,None,2021-01-05
2,"Presidente, buenos días. Judith Sánchez Reyes...",1,6,2021,Judith Sánchez Reyes Imagen Golfo Veracruz Ele...,"Judith Sánchez Reyes, corresponsal de Imagen d...","Judith Sánchez Reyes, Imagen del Golfo | Pedro...","Judith Sánchez Reyes, Imagen del Golfo","Pedro Villa,Caña, El Universal","Diego Elías Cedillo, diario Basta, Grupo Cantón",...,None,None,None,None,None,None,None,None,None,2021-01-06
3,"Buenos días, presidente. Shaila Rosagel, corr...",1,8,2021,Shaila Rosagel Grupo Healy El Imparcial Sonora...,"Gracias. Muchas gracias. Presidente, buenos ...","Shaila Rosagel, Grupo Healy, El Imparcial, La ...","Shaila Rosagel, Grupo Healy, El Imparcial, La ...","Daniel Marmolejo, La Cuarta República, Voces ...","Judith Sánchez Reyes, Imagen del Golfo",...,None,None,None,None,None,None,None,None,None,2021-01-08
4,La Secretaría de la Defensa Nacional informa ...,1,11,2021,La Defensa Nacional Aeropuerto Internacional F...,Si usted en particular va a aplicar a algún t...,"Paul Velázquez, Los Mochis | Francisco Huerta,...","Paul Velázquez, Los Mochis","Francisco Huerta, (Inaudible)",None,...,None,None,None,None,None,None,None,None,None,2021-01-11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
650,Gracias. Muy buenos días a todas y a todos. L...,12,19,2023,Muy Liliana Noble Pulso Saludable Time Out Par...,"Muy buenos días a todas y a todos. Gracias, d...","Liliana Noble, Pulso Saludable | Liliana Piña,...","Liliana Noble, Pulso Saludable","Liliana Piña, ATiempo.com.mx,PuenteLibre.mx","Fanny Martínez, La Verdad Noticias",...,None,None,None,None,None,None,None,None,None,2023-12-19
651,Continúan los esfuerzos del Gobierno de Méxic...,12,20,2023,Continúan Otis Guerrero Estamos A Tianguis Bie...,"Buenos días, señor presidente. ¿Se va a busca...","Andrés García, Códice 21 | Arturo Páramo, Grup...","Andrés García, Códice 21","Arturo Páramo, Grupo Imagen","Ángela Rodríguez, Radio Fórmula",...,None,None,None,None,None,None,None,None,None,2023-12-20
652,"La Secretaría de Infraestructura, Comunicacio...",12,21,2023,La Infraestructura Comunicaciones Transportes ...,"Buenos días, señor presidente. Con Agua Salud...","Fernando Olivas, Radio Relax,¿Qué Pasó?, digit...","Fernando Olivas, Radio Relax,¿Qué Pasó?, digital",Marisol Cruz Hernández,"Carlos Domínguez, Nación 14",...,None,None,None,None,None,None,None,None,None,2023-12-21
653,"Campaña de miedo en Tabasco. Políticos, comun...",12,27,2023,Campaña Tabasco Políticos Villahermosa Tabasco...,"Ernesto Ledesma, de Rompeviento Tv. Jefe de Go...","- José Sobrevilla, Noreste\n- Ernesto Ledesma,...","- José Sobrevilla, Noreste\n- Ernesto Ledesma,...",None,None,...,None,None,None,None,None,None,None,None,None,2023-12-27


In [93]:
# Step 1: Dynamically generate the list of reporter columns
value_vars = [col for col in final.columns if col.startswith('reporter')]

# Step 2: Pivot longer (wide to long format)
df_long = pd.melt(final, id_vars=['date'], value_vars=value_vars, 
                  var_name='reporter_type', value_name='reporter')

# Step 3: Filter out rows where 'reporter' is None
df_long = df_long[df_long['reporter'].notna()]

# Step 4: Drop the 'reporter_type' column if not needed
df_long = df_long.drop(columns=['reporter_type'])

In [94]:
df_long = df_long.sort_values(by='date').reset_index(drop=True)
print(df_long)

           date                                           reporter
0    2021-01-04  Héctor Tlatempa, Puntos Suspensivos Radio, Pun...
1    2021-01-04           Meme Yamel, The México News,Sin Censura 
2    2021-01-04                   Carlos Calzada, Radio Educación 
3    2021-01-04   Demian Duarte, Sonora Power, Lobos FM,Polític...
4    2021-01-04                       Nuri Fernández, La Caracola 
...         ...                                                ...
2307 2023-12-21                                    Ángela Buitrago
2308 2023-12-27  - José Sobrevilla, Noreste\n- Ernesto Ledesma,...
2309 2023-12-29             Beatriz Contreras, Gobierno de México 
2310 2023-12-29        Juan Hernández, Diario Basta, Grupo Cantón 
2311 2023-12-29                             Alberto Ruz, Lhuillier

[2312 rows x 2 columns]


In [92]:
# Step 1: Split the text column by commas
df_split = df_long['reporter'].str.split(',', expand=True)

# Step 2: Assign columns for reporter and outlets
df_long['reporter'] = df_split[0]  # The first column is the reporter
df_long['outlet1'] = df_split[1]  # The second column is the first outlet
df_long['outlet2'] = df_split[2]  # The third column is the second outlet (if exists)

# If you need more outlet columns, dynamically create them:
for i in range(3, df_split.shape[1]):
    df_long[f'outlet{i - 1}'] = df_split[i]

df_long

,date,reporter,outlet1,outlet2,outlet3,outlet4,outlet5,outlet6,outlet7,outlet8,...,outlet29,outlet30,outlet31,outlet32,outlet33,outlet34,outlet35,outlet36,outlet37,outlet38
0,2021-01-04,Héctor Tlatempa,Puntos Suspensivos Radio,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
1,2021-01-04,Meme Yamel,The México News,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2,2021-01-04,Carlos Calzada,Radio Educación,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
3,2021-01-04,Demian Duarte,Sonora Power,Política,RockandRoll Radio,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
4,2021-01-04,Nuri Fernández,La Caracola,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2307,2023-12-21,Ángela Buitrago,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2308,2023-12-27,- José Sobrevilla,Noreste\n- Ernesto Ledesma,LordMolécula,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2309,2023-12-29,Beatriz Contreras,Gobierno de México,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2310,2023-12-29,Juan Hernández,Diario Basta,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
